In [1]:
!pip install tensorflow scikit-learn pandas numpy boto3 GitPython PyGithub h5py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.7 MB/s eta 0:00:00


In [2]:
import os
import shutil
import base64
from datetime import datetime
import pandas as pd
import numpy as np

from git import Repo
from github import Github
import boto3

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

REPO SYNCHRONIZATION

In [5]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY

In [6]:
GITHUB_REPO = "anshul-1999/mds7-Anshul_Sharma"
BRANCH = "main"

LOCAL_REPO_PATH = "/content/mds7-Anshul_Sharma"
S3_BUCKET = "anshulmds1"
S3_PREFIX = "deeplearning/"

GITHUB_TARGET_DIR = "week-05-06-bigquery/deeplearning"
REGION = "eu-north-1"

ARTIFACTS_DIR = "/content/dl_artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

In [7]:
repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"

if os.path.exists(LOCAL_REPO_PATH):
    repo = Repo(LOCAL_REPO_PATH)
    repo.remotes.origin.pull(BRANCH)
else:
    repo = Repo.clone_from(repo_url, LOCAL_REPO_PATH, branch=BRANCH)

print("Repository synchronized.")

Repository synchronized.


DATA EXTRACTION & PREPROCESSING

In [9]:
csv_path = csv_path = "/content/mds7-Anshul_Sharma/week-03-04-powerbi/titanic_clean.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError("titanic_clean.csv not found in cloned repository.")

df = pd.read_csv(csv_path)
df.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,True,False,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,False,False,False
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,False,False,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,False,False,True
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,True,False,True


In [10]:
target_col = "Survived"

X = df.drop(columns=[target_col])
y = df[target_col]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

input_dim = X_train_scaled.shape[1]

In [11]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("Training set shape:", X_train_scaled.shape)
print("Test set shape:", X_test_scaled.shape)

print("\nFirst 5 rows of scaled training data:")
print(X_train_scaled[:5])

X shape: (891, 1579)
y shape: (891,)
Training set shape: (712, 1579)
Test set shape: (179, 1579)

First 5 rows of scaled training data:
[[ 0.96622201  0.82956755 -0.11207776 ...  0.         -0.03750293
  -0.03750293]
 [ 0.1461187  -0.37094484 -0.11207776 ...  0.         -0.03750293
  -0.03750293]
 [ 0.324909   -1.57145722 -0.11207776 ...  0.         -0.03750293
  -0.03750293]
 [ 1.59976154  0.82956755 -0.87980748 ...  0.         -0.03750293
  -0.03750293]
 [ 1.38987728 -0.37094484  0.11824116 ...  0.         -0.03750293
  -0.03750293]]


NEURAL NETWORK ENGINEERING & TRAINING

In [12]:
## Model A: 3-layer shallow network

model_3 = Sequential([
    Dense(16, activation="relu", input_shape=(input_dim,)),
    Dense(1, activation="sigmoid")
])

model_3.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_3 = model_3.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

loss_3, acc_3 = model_3.evaluate(X_test_scaled, y_test, verbose=0)

model_3_path = os.path.join(ARTIFACTS_DIR, "model_3_layers.h5")
model_3.save(model_3_path)

print("Model 3-layer accuracy:", acc_3)

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5870 - loss: 0.7968 - val_accuracy: 0.6434 - val_loss: 0.7863
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7610 - loss: 0.5187 - val_accuracy: 0.6294 - val_loss: 0.7491
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8787 - loss: 0.3516 - val_accuracy: 0.6573 - val_loss: 0.7250
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9297 - loss: 0.2426 - val_accuracy: 0.6503 - val_loss: 0.7081
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9666 - loss: 0.1728 - val_accuracy: 0.6364 - val_loss: 0.6999
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9859 - loss: 0.1203 - val_accuracy: 0.6434 - val_loss: 0.6925
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9965 - loss: 0.0850 - val_accuracy: 0.6503 - val_loss: 0.6863
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0605 - val_accuracy: 0.6434 - val_loss: 0.6824
Epo

Model 3-layer accuracy: 0.4189944267272949


In [13]:
## Model B: 5-layer deep network

model_5 = Sequential([
    Dense(64, activation="relu", input_shape=(input_dim,)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_5.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_5 = model_5.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

loss_5, acc_5 = model_5.evaluate(X_test_scaled, y_test, verbose=0)

model_5_path = os.path.join(ARTIFACTS_DIR, "model_5_layers.h5")
model_5.save(model_5_path)

print("Model 5-layer accuracy:", acc_5)

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6239 - loss: 0.6719 - val_accuracy: 0.5524 - val_loss: 0.6805
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8910 - loss: 0.3891 - val_accuracy: 0.6154 - val_loss: 0.6597
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9789 - loss: 0.1689 - val_accuracy: 0.6434 - val_loss: 0.6459
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9965 - loss: 0.0484 - val_accuracy: 0.6713 - val_loss: 0.6446
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9982 - loss: 0.0162 - val_accuracy: 0.6853 - val_loss: 0.6514
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0065 - val_accuracy: 0.6853 - val_loss: 0.6579
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 1.0000 - loss: 0.0036 - val_accuracy: 0.6923 - val_loss: 0.6645
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 0.0023 - val_accuracy: 0.6853 - val

Model 5-layer accuracy: 0.7262569665908813


AUTOMATED DOCUMENT GENERATION

In [14]:
readme_content = f"""
# Titanic Deep Learning ETLS Pipeline

## Project Overview
This project extends the Titanic classification pipeline from classical machine learning to deep learning using Keras Sequential neural networks.

## Dataset
The model uses the cleaned Titanic dataset: `titanic_clean.csv`.

## Preprocessing
- Target variable: `Survived`
- Feature matrix scaled using `StandardScaler`
- Train/test split: 80/20
- Random state: 42

## Model A: Shallow Network

**File:** `model_3_layers.h5`

Architecture:
- Dense hidden layer: 16 neurons, ReLU activation
- Output layer: 1 neuron, Sigmoid activation

Compilation:
- Optimizer: Adam
- Loss: Binary Crossentropy
- Metric: Accuracy

Test Accuracy: `{acc_3:.4f}`

## Model B: Deep Network

**File:** `model_5_layers.h5`

Architecture:
- Dense hidden layer 1: 64 neurons, ReLU activation
- Dense hidden layer 2: 32 neurons, ReLU activation
- Dense hidden layer 3: 16 neurons, ReLU activation
- Output layer: 1 neuron, Sigmoid activation

Compilation:
- Optimizer: Adam
- Loss: Binary Crossentropy
- Metric: Accuracy

Test Accuracy: `{acc_5:.4f}`

## Deployment
Artifacts were deployed to:
- AWS S3 bucket: `{S3_BUCKET}/{S3_PREFIX}`
- GitHub directory: `{GITHUB_TARGET_DIR}`

## Generated On
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

readme_path = os.path.join(ARTIFACTS_DIR, "README.md")

with open(readme_path, "w") as f:
    f.write(readme_content)

print("README.md generated.")

README.md generated.


In [15]:
notebook_path = "/content/Data_Science_Innovations_6_DeepL_ELTS_Pipeline.ipynb"